In [ ]:
from paths import LABEL_INPUT

red_veps = []
yellow_veps = []
#yellow_veps = ["AlphaMissense", "varity_R_LOO", "gMVP", "REVEL", "ESM1b", "CADD_raw", "Eigen-raw_coding", "Polyphen2_HVAR", "PrimateAI", "Polyphen2_HDIV", "SIFT"]

VEP_TYPE = {
    "AlphaMissense": "population-tuned",
    "BayesDel_addAF": "clinical-trained",
    "bayesdel_noAF": "clinical-trained",
    "cadd_raw": "clinical-trained",
    "clinpred": "clinical-trained",
    "dann": "clinical-trained",
    "deogen2": "clinical-trained",
    "eigen-raw_coding": "clinical-trained",
    "eigen-PC-raw_coding": "clinical-trained",
    "esm1b": "population-free",
    "fathmm-xf_coding": "clinical-trained",
    "gmvp": "clinical-trained",
    "list-s2": "population-tuned",
    "m-cap": "clinical-trained",
    "metalr": "clinical-trained",
    "metarnn": "clinical-trained",
    "metasvm": "clinical-trained",
    "mpc": "clinical-trained",
    "mutationassessor": "population-free",
    "mutformer": "clinical-trained",
    "mutpred": "clinical-trained",
    "MutScore": "clinical-trained",
    "mvp": "clinical-trained",
    "phactboost": "clinical-trained",
    "polyphen2_HVAR": "clinical-trained",
    "polyphen2_HDIV": "clinical-trained",
    "primateai": "population-tuned",
    "provean": "population-free",
    "revel": "clinical-trained",
    "sift": "population-free",
    "sift4g": "population-free",
    "varity_R": "clinical-trained",
    "varity_ER": "clinical-trained",
    "varity_R_LOO": "clinical-trained",
    "varity_ER_LOO": "clinical-trained",
    "vest4": "clinical-trained",
    "MAVEN": "population-free",
    "MAVEN_(average)":"population-free",
    "MutationTaster":"clinical-trained",
}

'''
VEP_TYPE = {
    "AlphaMissense": "a",
    "varity_R_LOO": "a",
    "gmvp": "a",
    "revel": "a",
    "esm1b": "a",
    "cadd_raw": "a",
    "eigen-raw_coding": "a",
    "polyphen2_HVAR": "a",
    "polyphen2_HDIV": "a",
    "primateai": "a",
    "sift": "a",
}
'''

import pandas as pd

df = pd.read_csv(LABEL_INPUT, sep="\t")

def map_category(vep):
    vep_lower = vep.lower()  # convert input to lowercase
    for k in VEP_TYPE:
        if k.lower() == vep_lower:  # convert key to lowercase
            return VEP_TYPE[k]
    return "Unknown"

df["Category"] = df["VEP"].apply(map_category)

COLOR_MAP = {
    "population-tuned": "#D62728",   # red
    "clinical-trained": "#FFD700",   # yellow
    "population-free": "#2CA02C",    # green
    "a": '#00BBFF'
}

import matplotlib.pyplot as plt

df = df.sort_values("ROCAUC", ascending=True)

bar_colors = []
bar_color = "#1f77b4"  # matplotlib default blue
for vep in df["VEP"]:
    if vep.lower() in [x.lower() for x in red_veps]:
        bar_colors.append("#D62728")
    elif vep.lower() in [x.lower() for x in yellow_veps]:
        bar_colors.append("#FFD700")
    else:
        bar_colors.append(bar_color)


# Mapping categories to background colors for y-axis labels
COLOR_MAP = {
    "population-tuned": "#D62728",   # red
    "clinical-trained": "#FFD700",   # yellow
    "population-free": "#2CA02C",    # green
    "a": '#00BBFF'
    #"Unknown": '#00BBFF'      ,  # optional gray for unknown
}

# Start plotting
plt.figure(figsize=(8, 10))
ax = plt.gca()

# Draw bars (all same color)
ax.barh(df["VEP"], df["ROCAUC"], color=bar_colors, edgecolor='none',zorder=2)

# Customize y-axis labels with background colors
for label in ax.get_yticklabels():
    vep_name = label.get_text()
    category = df.loc[df["VEP"] == vep_name, "Category"].values[0]
    label.set_bbox({
        "facecolor": COLOR_MAP.get(category, "#FFFFFF"),
        "edgecolor": "none",
        "pad": 0,
    })
    label.set_color("black")  # text color
    label.set_fontweight("bold")
    label.set_fontsize(18)


ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
#ax.set_xticks([0, 5, 10, 15])
#ax.set_xticks([0, 1, 2, 3, 4])
ax.grid(axis='x', which='major', linestyle='--', color='gray', alpha=0.5, zorder=1)

for spine in ax.spines.values():
    spine.set_visible(False)
plt.xlabel("ROC AUC")
#plt.xlabel("MWU -log10(pval)")
#plt.title("Performance of VEPs by Training Paradigm")

#plt.xlim(0, 4.1)
plt.xlim(0, 1.1)

plt.tight_layout()
plt.savefig("vep_auc_colored.png", dpi=300)
plt.show()